In [1]:
import numpy as np
import pandas as pd

from linearmodels import PooledOLS          # Pooled model
from linearmodels import RandomEffects      # Random-effect model
from linearmodels import PanelOLS           # Fixed-effect model
from linearmodels import FirstDifferenceOLS # First difference model

from linearmodels.panel import compare      # Compare the results of multiple models
from statsmodels.api import add_constant    # for matrices of regression design

In [2]:
df = pd.read_csv('Wages.csv')
df.head()

,exp,wks,bluecol,ind,south,smsa,married,sex,union,ed,black,lwage,id,time
0,3,32,no,0,yes,no,yes,male,no,9,no,5.56068,1,1
1,4,43,no,0,yes,no,yes,male,no,9,no,5.72031,1,2
2,5,40,no,0,yes,no,yes,male,no,9,no,5.99645,1,3
3,6,39,no,0,yes,no,yes,male,no,9,no,5.99645,1,4
4,7,42,no,1,yes,no,yes,male,no,9,no,6.06146,1,5


In [3]:
panel_df = df.set_index(['id', 'time'])
panel_df.head()

exp  wks bluecol  ind south smsa married   sex union  ed black  \
id time                                                                   
1  1       3   32      no    0   yes   no     yes  male    no   9    no   
   2       4   43      no    0   yes   no     yes  male    no   9    no   
   3       5   40      no    0   yes   no     yes  male    no   9    no   
   4       6   39      no    0   yes   no     yes  male    no   9    no   
   5       7   42      no    1   yes   no     yes  male    no   9    no   

           lwage  
id time           
1  1     5.56068  
   2     5.72031  
   3     5.99645  
   4     5.99645  
   5     6.06146

# 2.1 Модель 1

In [4]:
mod_pl = PooledOLS.from_formula(formula='lwage~1+ed+exp+I(exp**2)+wks', data=panel_df)
res_pl = mod_pl.fit()
res_pl

Dep. Variable:,lwage,R-squared:,0.2836
Estimator:,PooledOLS,R-squared (Between):,0.3163
No. Observations:,4165,R-squared (Within):,0.1957
Date:,"Wed, Mar 25 2026",R-squared (Overall):,0.2836
Time:,20:09:19,Log-likelihood,-1994.4
Cov. Estimator:,Unadjusted,,
,,F-statistic:,411.62
Entities:,595,P-value,0.0000
Avg Obs:,7.0000,Distribution:,"F(4,4160)"
Min Obs:,7.0000,,
Max Obs:,7.0000,F-statistic (robust):,411.62


P-value < 0.05. Регрессия значима

In [5]:
res_pl.f_statistic

Model F-statistic (homoskedastic)
H0: All parameters ex. constant are zero
Statistic: 411.6235
P-value: 0.0000
Distributed: F(4,4160)
WaldTestStatistic, id: 0x218064e4250

In [6]:
res_pl.wald_test(formula='exp = 0, I(exp ** 2)=0')

Linear Equality Hypothesis Test
H0: Linear equality constraint is valid
Statistic: 723.8557
P-value: 0.0000
Distributed: chi2(2)
WaldTestStatistic, id: 0x21808eb1650

P-value < 0.05. Влияние коэффициента значимо

In [7]:
res_pl = mod_pl.fit(cov_type='clustered', cluster_entity=True)

In [8]:
res_pl.f_statistic_robust

Model F-statistic (robust)
H0: All parameters ex. constant are zero
Statistic: 72.6817
P-value: 0.0000
Distributed: F(4,4160)
WaldTestStatistic, id: 0x2180873a890

In [9]:
res_pl.wald_test(formula='exp = 0, I(exp ** 2)=0')

Linear Equality Hypothesis Test
H0: Linear equality constraint is valid
Statistic: 166.6017
P-value: 0.0000
Distributed: chi2(2)
WaldTestStatistic, id: 0x21808ed5950

In [10]:
mod_re = RandomEffects.from_formula(formula='lwage~1+ed+exp+I(exp**2)+wks', data=panel_df)
res_re = mod_re.fit()
res_re

Dep. Variable:,lwage,R-squared:,0.4200
Estimator:,RandomEffects,R-squared (Between):,-1.1061
No. Observations:,4165,R-squared (Within):,0.5487
Date:,"Wed, Mar 25 2026",R-squared (Overall):,-0.6571
Time:,20:09:19,Log-likelihood,993.25
Cov. Estimator:,Unadjusted,,
,,F-statistic:,753.01
Entities:,595,P-value,0.0000
Avg Obs:,7.0000,Distribution:,"F(4,4160)"
Min Obs:,7.0000,,
Max Obs:,7.0000,F-statistic (robust):,753.01


In [11]:
res_re.f_statistic

Model F-statistic (homoskedastic)
H0: All parameters ex. constant are zero
Statistic: 753.0105
P-value: 0.0000
Distributed: F(4,4160)
WaldTestStatistic, id: 0x21808e960d0

In [12]:
res_re.wald_test(formula='exp = 0, I(exp ** 2)=0')

Linear Equality Hypothesis Test
H0: Linear equality constraint is valid
Statistic: 2890.4544
P-value: 0.0000
Distributed: chi2(2)
WaldTestStatistic, id: 0x21808f05f10

In [13]:
from scipy.stats import chi2

In [14]:
chi2.isf(0.05,df = 2)

np.float64(5.991464547107983)

In [15]:
res_re = mod_re.fit(cov_type='clustered', cluster_entity=True)

In [16]:
res_re.f_statistic_robust

Model F-statistic (robust)
H0: All parameters ex. constant are zero
Statistic: 400.1377
P-value: 0.0000
Distributed: F(4,4160)
WaldTestStatistic, id: 0x21808eea890

In [17]:
res_re.wald_test(formula='exp = 0, I(exp ** 2)=0')

Linear Equality Hypothesis Test
H0: Linear equality constraint is valid
Statistic: 1599.6203
P-value: 0.0000
Distributed: chi2(2)
WaldTestStatistic, id: 0x218066dcc50

In [18]:
mod_fe = PanelOLS.from_formula(formula='lwage~1+ed+exp+I(exp**2)+wks+EntityEffects', data=panel_df, drop_absorbed=True)
res_fe = mod_fe.fit()

C:\Users\stud.LIMM\AppData\Local\Temp\ipykernel_3924\445033095.py:2: AbsorbingEffectWarning: 
Variables have been fully absorbed and have removed from the regression:

ed

  res_fe = mod_fe.fit()


In [19]:
res_fe

Dep. Variable:,lwage,R-squared:,0.6566
Estimator:,PanelOLS,R-squared (Between):,-5.9083
No. Observations:,4165,R-squared (Within):,0.6566
Date:,"Wed, Mar 25 2026",R-squared (Overall):,-4.1270
Time:,20:09:19,Log-likelihood,2253.7
Cov. Estimator:,Unadjusted,,
,,F-statistic:,2273.7
Entities:,595,P-value,0.0000
Avg Obs:,7.0000,Distribution:,"F(3,3567)"
Min Obs:,7.0000,,
Max Obs:,7.0000,F-statistic (robust):,2273.7


In [20]:
res_fe.f_statistic

Model F-statistic (homoskedastic)
H0: All parameters ex. constant are zero
Statistic: 2273.7364
P-value: 0.0000
Distributed: F(3,3567)
WaldTestStatistic, id: 0x21808ef4550

In [21]:
res_fe.wald_test(formula='exp = 0, I(exp ** 2)=0')

Linear Equality Hypothesis Test
H0: Linear equality constraint is valid
Statistic: 6818.3689
P-value: 0.0000
Distributed: chi2(2)
WaldTestStatistic, id: 0x21808f43610

In [22]:
res_fe = mod_fe.fit(cov_type='clustered', cluster_entity=True)

In [23]:
res_fe.f_statistic_robust

Model F-statistic (robust)
H0: All parameters ex. constant are zero
Statistic: 1061.2464
P-value: 0.0000
Distributed: F(3,3567)
WaldTestStatistic, id: 0x21808f1b910

In [24]:
res_fe.wald_test(formula='exp = 0, I(exp ** 2)=0')

Linear Equality Hypothesis Test
H0: Linear equality constraint is valid
Statistic: 3183.5025
P-value: 0.0000
Distributed: chi2(2)
WaldTestStatistic, id: 0x21808f43e90

# 2.2 Модель 2.

In [25]:
mod_pl = PooledOLS.from_formula(formula='lwage~1+ed+exp+I(exp**2)+wks+south+smsa', data=panel_df)
res_pl = mod_pl.fit()
res_pl

Dep. Variable:,lwage,R-squared:,0.3102
Estimator:,PooledOLS,R-squared (Between):,0.3602
No. Observations:,4165,R-squared (Within):,0.1758
Date:,"Wed, Mar 25 2026",R-squared (Overall):,0.3102
Time:,20:09:19,Log-likelihood,-1915.6
Cov. Estimator:,Unadjusted,,
,,F-statistic:,311.60
Entities:,595,P-value,0.0000
Avg Obs:,7.0000,Distribution:,"F(6,4158)"
Min Obs:,7.0000,,
Max Obs:,7.0000,F-statistic (robust):,311.60


P-value < 0.05. Регрессия значима

In [26]:
res_pl.f_statistic

Model F-statistic (homoskedastic)
H0: All parameters ex. constant are zero
Statistic: 311.5953
P-value: 0.0000
Distributed: F(6,4158)
WaldTestStatistic, id: 0x21808f75710

In [27]:
res_pl.wald_test(formula='exp = 0, I(exp ** 2)=0')

Linear Equality Hypothesis Test
H0: Linear equality constraint is valid
Statistic: 686.5445
P-value: 0.0000
Distributed: chi2(2)
WaldTestStatistic, id: 0x21808f7e610

In [28]:
res_pl.wald_test(formula='south[T.yes] = 0, smsa[T.yes]=0')

Linear Equality Hypothesis Test
H0: Linear equality constraint is valid
Statistic: 160.3887
P-value: 0.0000
Distributed: chi2(2)
WaldTestStatistic, id: 0x21808f7d990

P-value < 0.05. Влияние коэффициента значимо

In [29]:
res_pl = mod_pl.fit(cov_type='clustered', cluster_entity=True)

In [30]:
res_pl.f_statistic_robust

Model F-statistic (robust)
H0: All parameters ex. constant are zero
Statistic: 61.5084
P-value: 0.0000
Distributed: F(6,4158)
WaldTestStatistic, id: 0x21808f5a750

In [31]:
res_pl.wald_test(formula='exp = 0, I(exp ** 2)=0')

Linear Equality Hypothesis Test
H0: Linear equality constraint is valid
Statistic: 162.2864
P-value: 0.0000
Distributed: chi2(2)
WaldTestStatistic, id: 0x21808f46ed0

In [32]:
res_pl.wald_test(formula='south[T.yes] = 0, smsa[T.yes]=0')

Linear Equality Hypothesis Test
H0: Linear equality constraint is valid
Statistic: 31.2556
P-value: 0.0000
Distributed: chi2(2)
WaldTestStatistic, id: 0x21808f47190